# 12. 用户查询太模糊？通过查询扩展，提升语义匹配能力

## 一、为什么要在 RAG 中进行 Query Expansion？

在构建基于 **RAG（Retrieval-Augmented Generation）** 的问答系统时，用户输入往往存在以下几个常见问题：

- 表达模糊、不完整或口语化  
- 缺乏上下文信息  
- 难以准确命中知识库中的相关文档  

如果直接使用用户的原始查询进行向量检索，可能会导致以下问题：

- 召回结果不足  
- 命中无关内容  
- 最终生成的答案不够准确或全面  

因此，在执行检索前对用户的问题进行 **语义级扩展与改写（Query Expansion / Rewriting）**，是一种有效的优化手段。

### 示例：模糊查询示例

面对如下模糊的用户提问：

- “我想去好玩的地方”  
- “有没有好吃的”  
- “适合亲子游的地方”  

这些问题缺乏具体的背景信息和明确的需求描述，直接检索往往难以命中关键内容。

### 解决方案

为了解决这一问题，可以采用以下两种策略来增强检索效果：

1. **问题改写（Query Rewriting）**  
   将模糊问题转化为更清晰、具体的问题，提升语义表达能力。

2. **多步骤检索（Multi-step Querying）**  
   将复杂问题拆解为多个子任务，逐步检索后整合答案，提高检索的全面性和准确性。

## 二 优化手段
### 2.1 在检索前进行问题改写（Query Rewriting）

安装依赖

In [1]:
! uv add langchain faiss-cpu transformers torch sentence-transformers dashscope langchain-community "unstructured[md]"

Resolved 216 packages in 1ms
Audited 196 packages in 9ms


In [1]:
import logging
from langchain_community.document_loaders import DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import OpenAIEmbeddings
from langchain_classic.chains import RetrievalQA
from langchain_openai import ChatOpenAI
import os
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
base_url = os.getenv("OPENAI_BASE_URL")

# 配置日志
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# 配置
DOCUMENT_PATH = './data/' 
FILE_PATTERN = '*.md'
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200
# Embedding模型
EMBEDDING_MODEL = 'text-embedding-3-small'
LLM_MODEL = 'gpt-4o'  
TOP_K = 5

# 1. 初始化LLM
logging.info(f"正在初始化{LLM_MODEL}模型...")
llm = ChatOpenAI(
    model=LLM_MODEL,
    api_key=api_key,
    base_url=base_url,
    temperature=0.7,
    streaming=True
)

# 2. 加载文档
logging.info(f"从{DOCUMENT_PATH}加载文档...")
loader = DirectoryLoader(DOCUMENT_PATH, glob=FILE_PATTERN)
docs = loader.load()
logging.info(f"成功加载{len(docs)}个文档")

# 3. 文档切片
logging.info("正在进行文档切片...")
text_spliiter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP
)
chunks = text_spliiter.split_documents(docs)
logging.info(f"文档切片完成，获得{len(chunks)}个文本块")

# 4. 初始化Embedding模型
logging.info(f"加载{EMBEDDING_MODEL}嵌入模型...")
embeddings = OpenAIEmbeddings(
    model=EMBEDDING_MODEL,
    api_key=api_key,
    base_url=base_url
)

# 5. 构建向量库
logging.info("构建FAISS向量库...")
vector_store = FAISS.from_documents(chunks, embeddings)
retriever = vector_store.as_retriever(search_kwargs={'k':TOP_K})

# 6. 创建RAG查询链
logging.info("创建RAG查询链...")
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever
)

logging.info("RAG系统初始化完成，ready for query!")

# 添加查询示例
def query_rag(question):
    logging.info(f"处理查询: {question}")
    result = qa_chain.invoke(question)
    return result['result']

d:\codes\agent_study\RAG性能调优\.venv\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.3.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(
d:\codes\agent_study\RAG性能调优\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-04-01 14:52:01,335 - INFO - 正在初始化gpt-4o模型...
2026-04-01 14:52:01,927 - INFO - 从./data/加载文档...
2026-04-01 14:52:02,229 - WARNING - libmagic is unavailable but assists in filetype detection. Please consider installing libmagic for better results.
2026-04-01 14:52:04,196 - WARNING - libmagic is unavailable but assists in filetype detection. Please consider installing libmagic for better results.
2026-04-01 14:52:04,249 - WARNING - libmagic is unavailable but assists in filetype detection. Please consider installing

传统检索方式（无优化）

In [2]:
# 测试查询
if __name__ == "__main__":
    test_question = "我想去好玩的地方?"
    answer = query_rag(test_question)
    print(f"问题: {test_question}\n答案: {answer}")

2026-04-01 14:52:20,968 - INFO - 处理查询: 我想去好玩的地方?
2026-04-01 14:52:22,790 - INFO - HTTP Request: POST https://api.vectorengine.ai/v1/embeddings "HTTP/1.1 200 OK"
2026-04-01 14:52:24,898 - INFO - HTTP Request: POST https://api.vectorengine.ai/v1/chat/completions "HTTP/1.1 200 OK"


问题: 我想去好玩的地方?
答案: 如果你想去好玩的地方，可以根据你的兴趣和所在的城市选择适合的景点：

1. **如果你在北京：**
   - **北京欢乐谷**：适合4岁以上，有过山车、水上乐园和亲子主题区“糖果王国”。
   - **北京环球影城**：适合5岁以上，有哈利波特魔法世界、变形金刚基地、小黄人乐园等。
   - **北京动物园**：适合全年龄段，有大熊猫馆和海洋馆，亲子设施齐全。

2. **如果你在上海：**
   - **上海迪士尼乐园**：适合3岁以上，有米奇童话专列、夜间烟花秀等，非常适合家庭游玩。
   - **上海海洋水族馆**：适合2岁以上，有亚洲最长的海底隧道，可以近距离观赏海洋生物。
   - **上海野生动物园**：全年龄段适合，可以乘坐观光车近距离接触动物。

3. **如果你在成都：**
   - **成都大熊猫繁育研究基地**：适合全年龄段，可以近距离观看大熊猫，早上8-10点是最佳时间。

如果你告诉我更多关于你的兴趣或者出行计划，我可以帮你推荐更具体的地方！


#### 2.1.1 使用 LLM 改写问题（增强语义表达）
Step 1: 我们首先定义一个 Prompt 模板，用于引导 LLM 对用户问题进行改写：

Step 2: 执行改写

In [8]:
from langchain_classic.chains import LLMChain
from langchain_core.prompts import PromptTemplate
import logging

rewrite_prompt = PromptTemplate(
    input_variables=["question"],
    template="""
你是一个旅游助手，请将以下用户问题改写为更清晰、完整的形式：

原始问题：
{question}

请输出一个改写后的问题，要求：
- 更具体
- 包含旅行类型（亲子/情侣/自驾等）
- 能帮助系统准确检索相关信息

输出格式：
[改写后] - <你的回答>
"""
)

def create_rewrite_chain(llm):
    return LLMChain(llm=llm, prompt=rewrite_prompt)

# 执行问题改写
def rewrite_question(rewrite_chain, question): 
    logging.info(f"正在改写问题: {question}")
    try:
        # 2. 修复调用方式：使用 predict 直接传入变量并获取字符串结果
        rewritten_question = rewrite_chain.predict(question=question).strip()
        
        if not rewritten_question:
            logging.warning("改写后的问题为空")
            return "未获得有效的改写结果"
        return rewritten_question
        
    except Exception as e:
        # 3. 优化调试：把真实的报错内容打印出来，避免被掩盖
        logging.error(f"改写问题时出错: {repr(e)}")
        return "改写问题时发生错误"

# 测试执行
rewrite_chain = create_rewrite_chain(llm) # 确保这里的 llm 已经正确初始化
test_question = "我想去好玩的地方?"
rewritten = rewrite_question(rewrite_chain, test_question)
print(f"改写后问题: {rewritten}")

2026-04-01 14:55:32,539 - INFO - 正在改写问题: 我想去好玩的地方?
2026-04-01 14:55:34,632 - INFO - HTTP Request: POST https://api.vectorengine.ai/v1/chat/completions "HTTP/1.1 200 OK"


改写后问题: [改写后] - 我想寻找适合亲子游的好玩地方，可以推荐一些适合带孩子一起体验的景点或活动吗？


In [9]:
answer = query_rag(rewritten)
print(f"问题: {rewritten}\n答案: {answer}")

2026-04-01 14:55:42,233 - INFO - 处理查询: [改写后] - 我想寻找适合亲子游的好玩地方，可以推荐一些适合带孩子一起体验的景点或活动吗？
2026-04-01 14:55:43,804 - INFO - HTTP Request: POST https://api.vectorengine.ai/v1/embeddings "HTTP/1.1 200 OK"
2026-04-01 14:55:46,067 - INFO - HTTP Request: POST https://api.vectorengine.ai/v1/chat/completions "HTTP/1.1 200 OK"


问题: [改写后] - 我想寻找适合亲子游的好玩地方，可以推荐一些适合带孩子一起体验的景点或活动吗？
答案: 当然可以！以下是几个适合亲子游的好玩地点和活动推荐：

### 上海
1. **上海迪士尼乐园**
   - 适合年龄：3岁以上
   - 推荐亮点：米奇童话专列、创极速光轮、夜间烟花秀，整体沉浸体验极佳。
   - 亲子设施：婴儿车租赁、哺乳室、儿童游乐区。
   - 附近餐厅：园内有米奇好朋友餐厅和唐老鸭亲子餐厅，步行5分钟可到迪士尼小镇的海底捞和外婆家。

2. **上海海洋水族馆**
   - 适合年龄：2岁以上
   - 推荐亮点：亚洲最长海底隧道，鲨鱼、魔鬼鱼、企鹅等近距离观赏。
   - 亲子设施：儿童讲解服务、婴儿车通道。
   - 附近餐厅：正大广场内有麦当劳、外婆家等儿童友好餐厅。

3. **上海野生动物园**
   - 适合年龄：全年龄段
   - 推荐亮点：乘坐观光车穿越自驾区，近距离观察长颈鹿、老虎等动物。
   - 亲子设施：喂食体验区、儿童骑马、婴儿车租赁。
   - 附近餐厅：园内熊猫餐厅，周边有多家农家乐。

---

### 北京
1. **北京环球影城**
   - 适合年龄：5岁以上
   - 推荐亮点：哈利波特魔法世界、小黄人乐园、变形金刚基地。
   - 亲子设施：婴儿车存放处、亲子洗手间。
   - 附近餐厅：园内三把扫帚餐厅，园外环球城市大道和通州万达广场的多家餐厅。

2. **北京动物园**
   - 适合年龄：全年龄段
   - 推荐亮点：大熊猫馆、非洲动物区、海洋馆。
   - 亲子设施：儿童喂食区、婴儿车租赁。
   - 附近餐厅：熊猫餐厅、悦来餐厅，步行可至西直门凯德MALL的必胜客和海底捞。

3. **北京欢乐谷**
   - 适合年龄：4岁以上
   - 推荐亮点：过山车、水上乐园、糖果王国亲子主题区。
   - 附近餐厅：园内美食街以及附近的麦当劳和必胜客。

---

### 成都
1. **成都大熊猫繁育研究基地**
   - 适合年龄：全年龄段
   - 推荐亮点：近距离观察大熊猫，早上8-10点时它们最活跃。
   - 亲子设施：儿童讲解志愿服务、婴儿车租赁。
   - 附近餐厅：基地内竹韵餐厅提供熊猫造型便当，周边龙潭路的川菜馆适合家庭聚餐。

这些地点不仅孩子们会玩得开心，家长

检索结果更加聚焦于“亲子游”相关内容，有效提升了准确性。

### 2.2 多步骤检索（Multi-step Querying）

对于涉及多个需求的复杂问题，单一检索往往难以覆盖所有方面。此时，我们可以将其拆分为多个子问题，分别进行检索后再综合结果。

示例场景：查找适合亲子游的景点及周边美食

处理流程如下：
1. 先检索“亲子友好型景点”；
2. 再根据这些景点，检索“附近的推荐餐厅”；
3. 最后将两次检索结果整合，形成完整的回答。

这种分阶段检索的方式能够显著提升结果的准确性和全面性，尤其适用于涉及多个维度的复合型查询。

In [10]:
from langchain_core.tools import Tool
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import RetrievalQA
from langchain_classic.agents import AgentExecutor, create_react_agent
from langchain_classic import hub


def search_child_friendly_attractions(query):
    try:
        logging.info(f"正在检索亲子友好型景点: {query}")
        result = qa_chain.invoke(query)["result"]
        if not result.strip(): 
            logging.warning("未检索到亲子友好型景点的结果")
            return "未找到相关亲子友好型景点"
        return result
    except Exception as e:
        logging.error(f"检索亲子友好型景点时出错: {str(e)}")
        return "检索亲子友好型景点时发生错误"
    
def search_nearby_restaurants(query):
    try:
        logging.info(f"正在检索附近的推荐餐厅: {query}")
        result = qa_chain.invoke(query)["result"]
        if not result.strip(): 
            logging.warning("未检索到附近餐厅的结果")
            return "未找到附近的推荐餐厅"
        return result
    except Exception as e:
        logging.error(f"检索附近餐厅时出错: {str(e)}")
        return "检索附近餐厅时发生错误"

def initialize_tools():
    return [
        Tool(
            name="SearchChildFriendlyAttractions",
            func=search_child_friendly_attractions,
            description="用于检索亲子友好型景点，输入应该是与亲子游玩相关的问题"
        ),
        Tool(
            name="SearchNearbyRestaurants",
            func=search_nearby_restaurants,
            description="用于根据景点检索附近的推荐餐厅，输入应该包含景点名称"
        )
    ]

def initialize_search_agent(llm):
    tools = initialize_tools()

    prompt = hub.pull("hwchase17/react")

    agent = create_react_agent(llm, tools, prompt)

    agent_executor = AgentExecutor(
        agent=agent, 
        tools=tools, 
        verbose=True,
        handle_parsing_errors=True # 推荐加上，防止大模型输出格式不对导致崩溃
    )

    return agent_executor

def run_multi_step_search(agent_executor, query):
    try:
        logging.info(f"开始执行多步检索: {query}")
        response = agent_executor.invoke({"input":query})
        formatted_response = f"🔁 多步检索结果:\n{response['output']}"
        print(formatted_response)
        return formatted_response
    except Exception as e:
        logging.error(f"执行多步检索时出错: {str(e)}")
        print("执行多步检索时发生错误，请检查日志")
        return None
    
if __name__ == "__main__":
   agent_executor = initialize_search_agent(llm)
   print(agent_executor)
   multi_step_query = """
我想找一个适合带孩子玩的地方，附近最好有好吃的餐厅。
先查有哪些亲子友好景点，再找附近的餐饮推荐。
"""
   run_multi_step_search(agent_executor, multi_step_query)

2026-04-01 15:00:05,778 - INFO - 开始执行多步检索: 
我想找一个适合带孩子玩的地方，附近最好有好吃的餐厅。
先查有哪些亲子友好景点，再找附近的餐饮推荐。



verbose=True agent=RunnableAgent(runnable=RunnableAssign(mapper={
  agent_scratchpad: RunnableLambda(lambda x: format_log_to_str(x['intermediate_steps']))
})
| PromptTemplate(input_variables=['agent_scratchpad', 'input'], input_types={}, partial_variables={'tools': 'SearchChildFriendlyAttractions(query) - 用于检索亲子友好型景点，输入应该是与亲子游玩相关的问题\nSearchNearbyRestaurants(query) - 用于根据景点检索附近的推荐餐厅，输入应该包含景点名称', 'tool_names': 'SearchChildFriendlyAttractions, SearchNearbyRestaurants'}, metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'}, template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the actio

2026-04-01 15:00:07,340 - INFO - HTTP Request: POST https://api.vectorengine.ai/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-01 15:00:07,378 - INFO - 正在检索亲子友好型景点: 适合带孩子玩的地方有哪些


Action: SearchChildFriendlyAttractions  
Action Input: 适合带孩子玩的地方有哪些

2026-04-01 15:00:09,035 - INFO - HTTP Request: POST https://api.vectorengine.ai/v1/embeddings "HTTP/1.1 200 OK"
2026-04-01 15:00:12,192 - INFO - HTTP Request: POST https://api.vectorengine.ai/v1/chat/completions "HTTP/1.1 200 OK"


...

这里列出了多个适合亲子游玩的地方，并且每个地方都附带附近的推荐餐厅。以下是推荐方案：

### 如果你在上海：
- **上海迪士尼乐园**：适合3岁以上儿童，有丰富的游乐设施和夜间烟花秀，附近餐厅包括米奇好朋友餐厅、唐老鸭亲子餐厅。
- **上海海洋水族馆**：适合2岁以上儿童，亮点是亚洲最长海底隧道，附近推荐去正大广场的外婆家或麦当劳。
- **上海野生动物园**：全年龄段都适合，推荐附近的熊猫餐厅或南六公路沿线农家乐。

### 如果你在北京：
- **北京环球影城**：适合5岁以上儿童，有哈利波特魔法世界和小黄人乐园，附近推荐三把扫帚餐厅或环球城市大道上的餐厅。
- **北京动物园**：全年龄段适合，推荐熊猫餐厅或西直门凯德MALL的餐厅如海底捞。
- **北京欢乐谷**：适合4岁以上儿童，园内有美食街，也可以选择麦当劳或必胜客。

### 如果你在成都：
- **成都大熊猫繁育研究基地**：全年龄段都适合，推荐附近的竹韵餐厅或龙潭路川菜馆。

你可以根据所在城市和孩子的年龄选择适合的地方及餐厅。如果需要更详细的信息，可以告诉我具体地点或需求。Invalid Format: Missing 'Action:' after 'Thought:'

2026-04-01 15:00:23,820 - INFO - HTTP Request: POST https://api.vectorengine.ai/v1/chat/completions "HTTP/1.1 200 OK"


Apologies for the confusion earlier. Let me summarize the recommendations based on the observations:

### 推荐亲子游玩的地点和附近餐厅：
1. **上海迪士尼乐园**
   - 适合年龄：3岁以上
   - 推荐餐厅：米奇好朋友餐厅、唐老鸭亲子餐厅、海底捞（迪士尼店）

2. **上海海洋水族馆**
   - 适合年龄：2岁以上
   - 推荐餐厅：正大广场外婆家、麦当劳

3. **北京环球影城**
   - 适合年龄：5岁以上
   - 推荐餐厅：三把扫帚餐厅、环球城市大道餐厅（如海底捞、西贝莜面村）

如果需要更具体的餐饮推荐，请告诉我选择的亲子景点！Invalid Format: Missing 'Action:' after 'Thought:'

2026-04-01 15:00:26,336 - INFO - HTTP Request: POST https://api.vectorengine.ai/v1/chat/completions "HTTP/1.1 200 OK"


I now know the final answer.

Final Answer: 根据你的需求，我推荐以下亲子友好景点和附近餐厅：

### 上海
1. **上海迪士尼乐园**
   - 适合年龄：3岁以上
   - 推荐餐厅：米奇好朋友餐厅、唐老鸭亲子餐厅、海底捞（迪士尼店）

2. **上海海洋水族馆**
   - 适合年龄：2岁以上
   - 推荐餐厅：正大广场外婆家、麦当劳

3. **上海野生动物园**
   - 适合年龄：全年龄段
   - 推荐餐厅：熊猫餐厅、南六公路沿线农家乐

### 北京
1. **北京环球影城**
   - 适合年龄：5岁以上
   - 推荐餐厅：三把扫帚餐厅、环球城市大道餐厅（如海底捞、西贝莜面村）

2. **北京动物园**
   - 适合年龄：全年龄段
   - 推荐餐厅：熊猫餐厅、西直门凯德MALL（如必胜客、海底捞）

3. **北京欢乐谷**
   - 适合年龄：4岁以上
   - 推荐餐厅：园内美食街、麦当劳、必胜客（朝阳路沿线）

### 成都
1. **成都大熊猫繁育研究基地**
   - 适合年龄：全年龄段
   - 推荐餐厅：竹韵餐厅（熊猫主题套餐）、龙潭路川菜馆（可调整辣度）

根据你所在的城市和孩子年龄选择合适的地点及餐厅，如果需要更详细的推荐或帮助，请告诉我！

> Finished chain.
🔁 多步检索结果:
根据你的需求，我推荐以下亲子友好景点和附近餐厅：

### 上海
1. **上海迪士尼乐园**
   - 适合年龄：3岁以上
   - 推荐餐厅：米奇好朋友餐厅、唐老鸭亲子餐厅、海底捞（迪士尼店）

2. **上海海洋水族馆**
   - 适合年龄：2岁以上
   - 推荐餐厅：正大广场外婆家、麦当劳

3. **上海野生动物园**
   - 适合年龄：全年龄段
   - 推荐餐厅：熊猫餐厅、南六公路沿线农家乐

### 北京
1. **北京环球影城**
   - 适合年龄：5岁以上
   - 推荐餐厅：三把扫帚餐厅、环球城市大道餐厅（如海底捞、西贝莜面村）

2. **北京动物园**
   - 适合年龄：全年龄段
   - 推荐餐厅：熊猫餐厅、西直门凯德MALL（如必胜客、海底捞）

3. **北京欢乐谷**
   - 适合年龄：4岁以上
   - 推荐餐厅：园内美食街、麦当劳、

优势： 分阶段检索，提高准确性和全面性。

## 2.3 效果对比与评估

为了验证优化策略的有效性，建议构建一个小规模的 Ground Truth 数据集 ，并设计简单的评估指标进行对比分析。

评估方法示例：
- 构建包含原始问题、改写后问题、期望答案的数据集；
- 判断最终生成的回答是否包含正确答案；
- 统计准确率或召回率等指标。

In [22]:
import logging
from typing import List, Dict, Tuple
from difflib import SequenceMatcher

def setup_logging():
    logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
setup_logging()

class GroundTruthDataset:
    """管理包含原始问题、改写后问题和期望答案的数据集"""
    def __init__(self):
        self.data: List[Dict[str, str]] = []
    
    def add_example(self, original_q: str, rewritten_q: str, ground_truth: str):
        """添加数据示例"""
        self.data.append({
            'original_q': original_q,
            'rewritten_q': rewritten_q,
            'ground_truth': ground_truth
        })
        logging.info("成功添加新的评估示例")
    
    def get_all_examples(self) -> List[Dict[str, str]]:
        """获取所有数据示例"""
        return self.data

class Evaluator:
    """评估模型回答的性能"""
    def __init__(self, similarity_threshold: float = 0.8):
        self.similarity_threshold = similarity_threshold
    
    def _is_answer_correct(self, answer: str, ground_truth: str) -> bool:
        """使用相似度判断答案是否正确"""
        similarity = SequenceMatcher(None, answer.lower(), ground_truth.lower()).ratio()
        return similarity >= self.similarity_threshold
    
    def evaluate_single(self, original_q: str, rewritten_q: str, answer: str, ground_truth: str) -> Tuple[int, Dict[str, str]]:
        """评估单个示例"""
        score = 1 if self._is_answer_correct(answer, ground_truth) else 0
        result = {
            'original_q': original_q,
            'rewritten_q': rewritten_q,
            'answer': answer,
            'ground_truth': ground_truth,
            'is_correct': score == 1
        }
        logging.info(f"原始问题: {original_q}\n改写问题: {rewritten_q}\n正确答案包含: {'✅' if score else '❌'}")
        return score, result
    
    def evaluate_dataset(self, dataset: GroundTruthDataset, answer_generator) -> Dict[str, float]:
        """评估整个数据集"""
        total = len(dataset.get_all_examples())
        correct = 0
        results = []

        for example in dataset.get_all_examples():
            try:
                answer = answer_generator(example['rewritten_q'])
                score, result = self.evaluate_single(
                    example['original_q'],
                    example['rewritten_q'],
                    answer,
                    example['ground_truth']
                )
                correct += score
                results.append(result)
            except Exception as e:
                logging.error(f"评估示例时出错: {str(e)}")
                continue

        accuracy = correct / total if total > 0 else 0
        # 可以扩展计算召回率、精确率、F1值等指标
        return {
            'accuracy': accuracy,
            'correct_count': correct,
            'total_count': total,
            'results': results
        }
    
    def generate_report(self, metrics: Dict[str, float]) -> str:
        """生成评估报告"""
        report = f"评估报告:\n"
        report += f"- 总样本数: {metrics['total_count']}\n"
        report += f"- 正确样本数: {metrics['correct_count']}\n"
        report += f"- 准确率: {metrics['accuracy']:.2%}\n"
        return report
    

if __name__ == "__main__":
    dataset = GroundTruthDataset()
    dataset.add_example(
        original_q="我想带孩子玩",
        rewritten_q="推荐适合亲子游玩的目的地",
        ground_truth="推荐了上海迪士尼乐园"
    )

    # 模拟回答生成函数（实际使用时替换为真实的模型调用）
    def answer_generator(query: str) -> str:
        answer = query_rag(question=query)
        print(f"生成回答：{answer}\n\n")
        return answer
    
    evaluator = Evaluator(similarity_threshold=0.5)

    metrics = evaluator.evaluate_dataset(dataset, answer_generator)

    report = evaluator.generate_report(metrics)
    print(report)



2026-04-01 15:22:02,146 - INFO - 成功添加新的评估示例
2026-04-01 15:22:02,146 - INFO - 处理查询: 推荐适合亲子游玩的目的地
2026-04-01 15:22:17,925 - INFO - HTTP Request: POST https://api.vectorengine.ai/v1/embeddings "HTTP/1.1 200 OK"
2026-04-01 15:22:20,401 - INFO - HTTP Request: POST https://api.vectorengine.ai/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-01 15:22:25,854 - INFO - 原始问题: 我想带孩子玩
改写问题: 推荐适合亲子游玩的目的地
正确答案包含: ❌


生成回答：以下是适合亲子游玩的几个推荐目的地，根据不同城市和景点特点选择：

---

### **上海**
1. **上海迪士尼乐园**
   - **适合年龄**：3岁以上  
   - **亮点**：米奇童话专列、创极速光轮、夜间烟花秀，沉浸式体验丰富。
   - **亲子设施**：婴儿车租赁、哺乳室、儿童游乐区。
   - **附近餐厅**：
     - 园内：米奇好朋友餐厅（儿童套餐）、唐老鸭亲子餐厅。
     - 园外：迪士尼小镇的海底捞（儿童面塑服务）和外婆家（清淡江浙菜）。

2. **上海海洋水族馆**
   - **适合年龄**：2岁以上  
   - **亮点**：亚洲最长海底隧道，可近距离观赏鲨鱼、魔鬼鱼、企鹅等。
   - **亲子设施**：婴儿车通道、儿童讲解服务。
   - **附近餐厅**：
     - 陆家嘴正大广场美食城（步行5分钟）：外婆家、必胜客、麦当劳等适合儿童的餐厅。

3. **上海野生动物园**
   - **适合年龄**：全年龄段  
   - **亮点**：可乘坐观光车穿越自驾区，近距离接触长颈鹿、老虎等动物。
   - **亲子设施**：喂食体验区、儿童骑马区。
   - **附近餐厅**：
     - 园内：熊猫餐厅（动物造型的儿童套餐）。
     - 园外：南六公路沿线农家乐，适合家庭聚餐。

---

### **北京**
1. **北京环球影城**
   - **适合年龄**：5岁以上  
   - **亮点**：哈利波特魔法世界、变形金刚基地、小黄人乐园。
   - **亲子设施**：婴儿车存放处、儿童身高限制清晰标注。
   - **附近餐厅**：
     - 园内：三把扫帚餐厅（哈利波特主题）。
     - 园外：环球城市大道商业街的海底捞、西贝莜面村等。

2. **北京动物园**
   - **适合年龄**：全年龄段  
   - **亮点**：大熊猫馆、海洋馆、非洲动物区。
   - **亲子设施**：婴儿车租赁、儿童喂食区。
   - **附近餐厅**：
     - 园内：熊猫餐厅（熊猫主题套餐）。
     - 园外：西直门凯德mall的海底捞、必胜客等。

3. **北京欢乐谷**
   - **适合年龄**：4岁以上  
   - **亮点**：过山车、水上乐园、糖果王

2026-04-01 16:29:09,002 - WARNING - Failed to refresh cache entry hwchase17/react: Connection error caused failure to GET /commits/hwchase17/react/latest in LangSmith API. Please confirm your internet connection. ConnectionError(MaxRetryError('HTTPSConnectionPool(host=\'api.smith.langchain.com\', port=443): Max retries exceeded with url: /commits/hwchase17/react/latest (Caused by ReadTimeoutError("HTTPSConnectionPool(host=\'api.smith.langchain.com\', port=443): Read timed out. (read timeout=10.0)"))'))
Content-Length: None
API Key: lsv2_********************************************03
2026-04-01 16:30:52,230 - WARNING - Failed to refresh cache entry hwchase17/react: Connection error caused failure to GET /commits/hwchase17/react/latest in LangSmith API. Please confirm your internet connection. ConnectionError(MaxRetryError('HTTPSConnectionPool(host=\'api.smith.langchain.com\', port=443): Max retries exceeded with url: /commits/hwchase17/react/latest (Caused by ReadTimeoutError("HTTPSConn

通过此类评估，可以量化不同策略的效果差异，为后续优化提供数据支持。

### 总结

| 优化点 | 效果 |
| --- | --- |
| 问题改写 | 提高检索准确性，避免遗漏关键信息 |
| 多步骤检索 | 提升复杂问题的回答完整性 |
| 中文 Embedding | 提升向量召回质量 |
| 本地 LLM | 隐私安全，可控性强 |


### 三、更多优化建议

| 优化方法 | 收益点 |
| --- | --- |
| 多变体召回 + 聚合 | 将多个改写后的查询同时提交给向量库，聚合结果 |
| HyDE 策略 | 利用假设文档（Hypothetical Document）辅助检索 |
| 多路召回 + Reranking | 使用 rerank 模块对检索结果打分排序，提升精度 |
